In [8]:
import os
import shutil
import random
from pathlib import Path
import yaml

# =========================
# 설정
# =========================
DATASET_DIR = Path("dataset-cf")

IMG_TRAIN_DIR = DATASET_DIR / "images" / "train"
LBL_TRAIN_DIR = DATASET_DIR / "labels" / "train"

IMG_VAL_DIR = DATASET_DIR / "images" / "val"
LBL_VAL_DIR = DATASET_DIR / "labels" / "val"

VAL_RATIO = 0.2
SEED = 42

# 클래스명 수정 가능
CLASS_NAMES = [
    "person"
]

# =========================
# 폴더 생성
# =========================
IMG_VAL_DIR.mkdir(parents=True, exist_ok=True)
LBL_VAL_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)

# =========================
# 영상 이름 추출 함수
# 예:
# leg_01_000001.jpg -> leg_01
# pushup_03_frame_0001.jpg -> pushup_03
# =========================
def get_video_id(filename):
    stem = Path(filename).stem
    parts = stem.split("_")

    # 보통 video명_프레임번호 구조일 때
    # 마지막이 숫자면 제거
    if parts[-1].isdigit():
        return "_".join(parts[:-1])

    # frame_0001 같은 구조면 뒤 2개 제거
    if len(parts) >= 2 and parts[-2].lower() == "frame":
        return "_".join(parts[:-2])

    return stem

# =========================
# 이미지 목록 수집
# =========================
image_files = []
for ext in ["*.jpg", "*.jpeg", "*.png"]:
    image_files.extend(list(IMG_TRAIN_DIR.glob(ext)))

if not image_files:
    raise FileNotFoundError("dataset_cf/images/train 안에 이미지가 없습니다.")

# =========================
# 영상 단위로 그룹화
# =========================
video_groups = {}

for img_path in image_files:
    video_id = get_video_id(img_path.name)
    video_groups.setdefault(video_id, []).append(img_path)

video_ids = list(video_groups.keys())
random.shuffle(video_ids)

val_count = max(1, int(len(video_ids) * VAL_RATIO))
val_video_ids = set(video_ids[:val_count])

print("전체 영상 수:", len(video_ids))
print("검증용 영상 수:", len(val_video_ids))
print("검증용 영상:", sorted(val_video_ids))

# =========================
# val 영상에 해당하는 이미지/라벨 이동
# =========================
moved_img = 0
moved_lbl = 0

for video_id in val_video_ids:
    for img_path in video_groups[video_id]:
        label_path = LBL_TRAIN_DIR / f"{img_path.stem}.txt"

        new_img_path = IMG_VAL_DIR / img_path.name
        shutil.move(str(img_path), str(new_img_path))
        moved_img += 1

        if label_path.exists():
            new_label_path = LBL_VAL_DIR / label_path.name
            shutil.move(str(label_path), str(new_label_path))
            moved_lbl += 1
        else:
            print(f"라벨 없음: {label_path.name}")

print(f"\n이동된 이미지 수: {moved_img}")
print(f"이동된 라벨 수: {moved_lbl}")

# =========================
# data.yaml 생성
# =========================
data_yaml = {
    "path": str(DATASET_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",

    # YOLO Pose 필수
    "kpt_shape": [17, 3],

    # 좌우 뒤집기 시 관절 인덱스 매칭
    "flip_idx": [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15],

    "names": {
        0: "person"
    }
}

yaml_path = DATASET_DIR / "data.yaml"

with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml, f, allow_unicode=True, sort_keys=False)

print("\ndata.yaml 생성 완료:")
print(yaml_path)

전체 영상 수: 47
검증용 영상 수: 9
검증용 영상: ['leg_leg_angle_01', 'leg_leg_angle_02', 'lunge_lunge_01', 'lunge_lunge_wait_01', 'plank_plank_03', 'plank_plank_04', 'plank_plank_angle_01', 'plank_plank_dark_01', 'pushup_pushup_06']

이동된 이미지 수: 1917
이동된 라벨 수: 1917

data.yaml 생성 완료:
dataset-cf\data.yaml


In [ ]:
import os
from ultralytics import YOLO
import torch

device = 0 if torch.cuda.is_available() else "cpu"

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: CPU 사용 중")

# Small Pose 모델로 변경
model = YOLO("yolov8s-pose.pt")

model.train(
    data="dataset-cf/data.yaml",
    epochs=15,
    imgsz=640,
    batch=16,              
    device=device,
    workers=4,

    freeze=22,

    lr0=5e-4,              # freeze 학습이므로 낮은 학습률
    lrf=0.01,
    optimizer="AdamW",
    pretrained=True,

    patience=5,
    project="runs",
    name="yolov8s_pose_cf",
    plots=False
)

print("\n완료")

GPU: NVIDIA GeForce RTX 4060
New https://pypi.org/project/ultralytics/8.4.71 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.68  Python-3.12.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset-cf/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=22, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-pose.pt, momentum=0.937, mo

In [ ]:
import os
import torch
from ultralytics import YOLO
from config import DATA_YAML_PATH

device = 0 if torch.cuda.is_available() else "cpu"

# 데이터셋 YAML 경로
DATA_YAML_PATH = DATA_YAML_PATH

# =========================================================
# Phase 1: Head 정렬
# - COCO 사전학습 지식 보존
# - 새 데이터셋 라벨 구조에 빠르게 적응
# =========================================================

model = YOLO("yolov8s-pose.pt")

model.train(
    data=DATA_YAML_PATH,
    imgsz=640,
    batch=16,             
    device=device,
    workers=4,
    amp=True,

    freeze=22,            # backbone 고정
    epochs=10,            
    patience=5,

    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=2.0,
    weight_decay=0.0005,

    project="runs/pose",
    name="yolov8s_ph1_cf",
    pretrained=True,
    plots=True
)

phase1_weights = os.path.join(
    model.trainer.save_dir,
    "weights",
    "best.pt"
)

print("Phase 1 best.pt:", phase1_weights)


# =========================================================
# Phase 2: 본 파인튜닝
# - 일부 backbone 개방
# - pose 성능 중심으로 정밀 학습
# =========================================================

print("\n[Phase 2] YOLOv8m-pose 본 파인튜닝 시작")

model_fine = YOLO(phase1_weights)

model_fine.train(
    data=DATA_YAML_PATH,
    imgsz=640,
    batch=16,
    device=device,
    workers=4,
    amp=True,

    freeze=6,           
    epochs=60,
    patience=12,

    optimizer="AdamW",
    lr0=0.0003,
    lrf=0.01,
    warmup_epochs=3.0,
    weight_decay=0.001,

    box=6.0,
    pose=12.0,
    kobj=1.0,
    cls=0.5,

    mosaic=0.5,
    close_mosaic=10,
    fliplr=0.5,
    degrees=5.0,
    translate=0.08,
    scale=0.4,

    project="runs/pose",
    name="yolov8s_ph2_cf",
    pretrained=True,
    plots=True
)

print("\n학습 완료")
print("최종 가중치 위치:")
print(os.path.join(model_fine.trainer.save_dir, "weights", "best.pt"))

New https://pypi.org/project/ultralytics/8.4.72 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.68  Python-3.12.13 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\DS\Desktop\cv\dataset-cf\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=22, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-pose.pt, momentum=0.937, mosaic=1.